# Week 1 — 环境自检 (env_check.ipynb)

**目标**：2 分钟内验证 Colab 环境可用，GPU 可见，依赖版本锁定。

建议排程：周二晚 20:30–22:30 的前 30 分钟跑完这个 notebook。

## 1. 挂载 Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/transformer-anomaly'
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/data', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/models', exist_ok=True)
%cd {PROJECT_ROOT}
print('Project root:', os.getcwd())

ModuleNotFoundError: No module named 'google.colab'

## 2. GPU 检查

In [ ]:
!nvidia-smi || echo 'No GPU — 去 Runtime → Change runtime type → GPU'

import torch
print('\nPyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('Memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

## 3. 安装依赖（只首次运行需要）

In [ ]:
%%capture
# Colab 自带 torch/numpy/pandas，不重装；补装缺失的
!pip install -q kaggle==1.6.6 opendatasets==0.1.22 seaborn==0.13.1

## 4. 版本快照（记录下来，本周所有 notebook 用同版本）

In [ ]:
import torch, numpy, pandas, sklearn, matplotlib, seaborn
versions = {
    'torch': torch.__version__,
    'numpy': numpy.__version__,
    'pandas': pandas.__version__,
    'sklearn': sklearn.__version__,
    'matplotlib': matplotlib.__version__,
    'seaborn': seaborn.__version__,
}
for k, v in versions.items():
    print(f'{k:12s} = {v}')

## 5. 设定随机种子（以后所有 notebook 第一个 cell 就跑这段）

In [ ]:
import random, numpy as np, torch
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Seed: {SEED}, Device: {device}')

## 6. 快速 Sanity Check：矩阵乘法 + GPU 计算

如果能打印 shape 和耗时，说明环境完全 OK。

In [ ]:
import time
a = torch.randn(4096, 4096, device=device)
b = torch.randn(4096, 4096, device=device)
torch.cuda.synchronize() if device.type == 'cuda' else None
t0 = time.time()
c = a @ b
torch.cuda.synchronize() if device.type == 'cuda' else None
print(f'Matrix multiply (4096x4096): {(time.time()-t0)*1000:.1f} ms, result shape {c.shape}')

## ✅ 验收

如果以上 6 个 cells 都无报错，打勾下面清单：

- [ ] Drive 已挂载，`transformer-anomaly/` 目录创建
- [ ] GPU 型号显示（T4 或 L4 或更好）
- [ ] PyTorch 版本 ≥ 2.0
- [ ] Seed 设定成功
- [ ] Matrix multiply 耗时 < 200ms（GPU）或 < 5000ms（CPU）

接下来打开 `01_eda.ipynb` 做数据探索。